# Tome 10, Chapter 9: Martingales II (Python companion)

Exercises first, solutions at the end. We watch a martingale lose its mean, gamble at roulette, check de Finetti's theorem on Pólya's urn, measure Kakutani's dichotomy on biased coins, and put martingales to work on two computer science problems: **Thompson sampling** for choosing which version of a web page to show, and **anytime-valid A/B tests** that stay honest however often the analyst peeks. Requires `numpy` and `scipy`.

In [ ]:
import math
from fractions import Fraction
import numpy as np
from scipy import stats
from scipy.special import gammaln
rng = np.random.default_rng(90)

## Exercises

**P9.1 (★) A martingale that loses its mean.** Let $M_n=\prod_{k\le n}2\xi_k$ with fair bits $\xi_k$. Estimate $\mathbb EM_n$ by the average of $10^5$ simulated values, for $n=5,10,20,30$. Explain the results with uniform integrability.

**P9.2 (★) Roulette.** Betting one euro on red at European roulette wins with probability $p=\frac{18}{37}$. Estimate by simulation the probability of ever being $20$ euros ahead, and compare with $(p/q)^{20}$ from Example 9.6. (Stop a run when it reaches $+20$ or $-200$; why is the error from the second rule negligible?)

**P9.3 (★★) De Finetti and Pólya.** (a) For the urn starting with $a=2$ red and $b=3$ green balls, compute exactly, with fractions, the probability of every sequence of $n=6$ draws, and check that it equals $\int_0^1\theta^s(1-\theta)^{n-s}\beta_{2,3}(\theta)\,\mathrm d\theta$ (Exercise 9.4). (b) Run the urn $5000$ times for $2000$ steps and compare the final proportions of red with the Beta$(2,3)$ law (Kolmogorov–Smirnov test).

**P9.4 (★★) CS application: Thompson sampling.** A shop shows one of three page designs to each visitor; the purchase probabilities are $0.040$, $0.050$ and $0.060$, unknown to the shop. Over $20\,000$ visitors, compare the *regret* (expected purchases lost compared with always showing the best page) of: (a) a uniform split, (b) $\varepsilon$-greedy with $\varepsilon=0.1$, (c) Thompson sampling with uniform priors: for each visitor, draw $\Theta_j$ from the Beta posterior of each design and show the design with the largest draw. Average over $200$ runs. Check also that the posterior mean of the best design converges to $0.06$ (Theorem 9.12).

**P9.5 (★★) Kakutani's dichotomy.** Let $P$ be the fair coin and $Q$ the coin with bias $\frac12+\varepsilon_n$ at toss $n$. Under $Q$, compute $\log M_n$, the log-likelihood ratio $\log\frac{dQ}{dP}$ on the first $n$ tosses, for $n=10^2,\dots,10^6$, with (a) $\varepsilon_n=\frac14n^{-1/2}$ and (b) $\varepsilon_n=\frac14n^{-3/4}$. In which case does it go to $+\infty$ (so that $Q\perp P$)? Compare with $\prod\rho_n$.

**P9.6 (★★★) CS application: peeking at an A/B test.** Visitors arrive in pairs, one shown page A and one page B; keep only the pairs where exactly one of the two buys, and record $X=1$ if it was B. If the pages are equally good, the $X_i$ are fair coin tosses. (a) Under this null hypothesis, simulate $4000$ experiments of $10\,000$ pairs, and compute the fraction in which the classical two-sided test at level $5\%$, $|2S_n-n|\ge1.96\sqrt n$, rejects at *some* look, when the analyst looks every $1000$ pairs, every $100$ pairs, and after every pair. (b) Do the same for the betting test of Theorem 9.16 with $\alpha=0.05$. (c) If B is better, with $P(X=1)=0.55$, how many pairs does the betting test need (median and $90\%$ quantile of its stopping time)? Compare with the fixed sample size giving the classical test $80\%$ power.

## Solutions

### P9.1
**How to find it.** $M_n=2^n$ with probability $2^{-n}$ and $0$ otherwise, so $\mathbb EM_n=1$ for every $n$. A sample of size $10^5$ almost never contains the event once $2^n\gg10^5$.

In [ ]:
T = 10**5
for n in (5, 10, 20, 30):
    xi = rng.integers(0, 2, size=(T, n), dtype=np.int8)
    M = np.where(xi.all(axis=1), 2.0**n, 0.0)
    print(f"n = {n:2d}: sample mean of M_n = {M.mean():9.4f}   (true mean 1;  P(M_n > 0) = {2.0**-n:.1e})")

For $n=5$ and $n=10$ the sample mean is close to $1$; for $n=20$ the event $\{M_n>0\}$ has probability about $10^{-6}$ and is usually missed, so the sample mean is $0$ (or, once in ten runs, about $10$); for $n=30$ it is $0$. The whole mean lives on an event that the sample never sees: this is what failure of uniform integrability looks like in a simulation, and it is why Monte Carlo estimates of rare large payoffs are treacherous.

### P9.2
**How to find it.** From $-200$, the probability of ever climbing to $+20$ is $(p/q)^{220}\approx7\times10^{-6}$, by the same formula; ignoring it changes the answer by less than that.

In [ ]:
p = 18 / 37; q = 1 - p
runs = 20000
pos = np.zeros(runs, dtype=np.int64); alive = np.ones(runs, bool); hit = np.zeros(runs, bool)
while alive.any():
    idx = np.flatnonzero(alive)
    pos[idx] += np.where(rng.random(idx.size) < p, 1, -1)
    up = pos[idx] >= 20; down = pos[idx] <= -200
    hit[idx[up]] = True
    alive[idx[up | down]] = False
est = hit.mean(); se = math.sqrt(est * (1 - est) / runs)
print(f"simulated {est:.4f} +- {se:.4f};  (p/q)^20 = {(p/q)**20:.4f};  neglected (p/q)^220 = {(p/q)**220:.1e}")

The simulation agrees with $(18/19)^{20}\approx0.339$ within its standard error. One gambler in three will at some point be $20$ euros up, and two in three never will, although they may play forever.

### P9.3
**How to find it.** Follow the urn along each sequence with exact fractions; the Beta integral is $B(a+s,b+n-s)/B(a,b)$, computed with factorials.

In [ ]:
from itertools import product
a, b, n = 2, 3, 6
def seq_prob(x):
    r, g, pr = a, b, Fraction(1)
    for xi in x:
        if xi: pr *= Fraction(r, r + g); r += 1
        else:  pr *= Fraction(g, r + g); g += 1
    return pr
def B(x, y):   # Beta function at positive integers
    return Fraction(math.factorial(x - 1) * math.factorial(y - 1), math.factorial(x + y - 1))
ok = all(seq_prob(x) == B(a + sum(x), b + n - sum(x)) / B(a, b) for x in product((0, 1), repeat=n))
print("all", 2**n, "sequences match the Beta(2,3) mixture:", ok)
print("P(all six red) =", seq_prob((1,)*6), "=", B(a + 6, b) / B(a, b))

runs, steps = 5000, 2000
red = np.full(runs, a); tot = a + b
for t in range(steps):
    red += rng.random(runs) < red / tot; tot += 1
final = red / tot
print("KS test against Beta(2,3):", stats.kstest(final, stats.beta(a, b).cdf))
print("mean of final proportions:", final.mean().round(4), " Beta(2,3) mean:", a / (a + b))

All $64$ probabilities agree exactly with the mixture (for example $P(\text{six reds})=\frac{1}{30}$), and the final proportions pass the Kolmogorov–Smirnov test against Beta$(2,3)$ (the $p$-value is not small). After $2000$ steps the proportion is still at distance about $1/\sqrt{2000}$ from its limit, too little for the test to notice.

### P9.4
**How to find it.** Keep, for each design, the number of purchases $s_j$ and of failures $f_j$; the posterior with a uniform prior is Beta$(s_j+1,f_j+1)$. Regret is $\sum_t(0.06-p_{j_t})$.

In [ ]:
P = np.array([0.040, 0.050, 0.060]); best = P.max()
H, R = 20000, 200
def run(policy, rng):
    s = np.zeros(3); f = np.zeros(3); regret = 0.0
    for t in range(H):
        if policy == "uniform":
            j = t % 3
        elif policy == "egreedy":
            j = rng.integers(3) if rng.random() < 0.1 else int(np.argmax((s + 1) / (s + f + 2)))
        else:                                   # Thompson sampling
            j = int(np.argmax(rng.beta(s + 1, f + 1)))
        y = rng.random() < P[j]
        s[j] += y; f[j] += 1 - y
        regret += best - P[j]
    return regret, (s[2] + 1) / (s[2] + f[2] + 2), s + f
res = {}
for pol in ("uniform", "egreedy", "thompson"):
    out = [run(pol, np.random.default_rng(1000 + r)) for r in range(R)]
    reg = np.array([o[0] for o in out]); pm = np.array([o[1] for o in out]); pulls = np.mean([o[2] for o in out], axis=0)
    res[pol] = reg
    print(f"{pol:9s} regret {reg.mean():6.1f} +- {reg.std()/math.sqrt(R):4.1f}   "
          f"posterior mean of best design {pm.mean():.4f} (sd {pm.std():.4f})   visitors per design {pulls.round(0)}")

The uniform split loses $20\,000\times\frac{0.02+0.01}3=200$ purchases, as it must. $\varepsilon$-greedy loses much less, but its regret varies about twice as much from run to run: when the best design starts badly, greedy play may ignore it for a long time, and it keeps wasting $10\%$ of the visitors on random designs forever. Thompson sampling has the smallest regret here: it keeps exploring each design in proportion to the posterior probability that it is the best, and that probability goes to $0$ for the worse designs. The posterior mean of the best design is close to $0.06$, as Doob's theorem predicts; for the worse designs, which Thompson sampling rarely shows, the posteriors stay wide, and that is fine, since a business only needs to know they are worse.

### P9.5
**How to find it.** Under $Q$, $\log M_n=\sum_{k\le n}\bigl[X_k\log(1+2\varepsilon_k)+(1-X_k)\log(1-2\varepsilon_k)\bigr]$. Its mean under $Q$ is $\sum_kD(q_k\|p_k)\approx\sum2\varepsilon_k^2$, which diverges like $\frac18\log n$ in case (a) and converges in case (b). The affinities satisfy $\log\rho_k\approx-\varepsilon_k^2/2$.

In [ ]:
N = 10**6; k = np.arange(1, N + 1)
for name, eps in (("(a) n^-1/2", 0.25 * k**-0.5), ("(b) n^-3/4", 0.25 * k**-0.75)):
    logrho = np.log((np.sqrt(1 + 2 * eps) + np.sqrt(1 - 2 * eps)) / 2)
    paths = []
    for r in range(5):
        X = rng.random(N) < 0.5 + eps
        inc = np.where(X, np.log1p(2 * eps), np.log1p(-2 * eps))
        paths.append(np.cumsum(inc))
    print(name, "  log prod rho_k up to 1e6:", round(logrho.sum(), 3))
    for n in (10**2, 10**3, 10**4, 10**5, 10**6):
        print(f"   n = {n:7d}: log M_n under Q for 5 runs:", np.round([p[n - 1] for p in paths], 2))

In case (a), $\log M_n$ drifts upwards like $\frac18\log n$ (about $1.7$ at $n=10^6$), slowly but surely to $+\infty$, and $\log\prod\rho_k$ drifts to $-\infty$: the measures are singular, but the evidence accumulates only logarithmically, so a million tosses give a likelihood ratio of only about $e^{1.7}\approx5$. In case (b), $\log M_n$ settles at a random finite limit: the measures are equivalent, and no amount of data will ever decide between them with certainty.

### P9.6
**How to find it.** Simulate in blocks. The classical test rejects at look $n$ if $|2S_n-n|\ge1.96\sqrt n$. The betting test rejects as soon as $\log M_n=n\log2+\log S_n!+\log(n-S_n)!-\log(n+1)!\ge\log20$. For the fixed-sample comparison, the classical test with $n$ pairs has power about $\Phi\bigl(\frac{(2\theta-1)\sqrt n-1.96}{2\sqrt{\theta(1-\theta)}}\bigr)$.

In [ ]:
def simulate(theta, runs=4000, n=10000, block=500):
    looks = {"every 1000": np.arange(1000, n + 1, 1000), "every 100": np.arange(100, n + 1, 100), "every pair": np.arange(1, n + 1)}
    rej = {k: 0 for k in looks}; bet_rej = 0; stop_times = []
    nn = np.arange(1, n + 1)
    for start in range(0, runs, block):
        X = (rng.random((block, n)) < theta).astype(np.int32)
        S = np.cumsum(X, axis=1)
        z = np.abs(2 * S - nn) >= 1.96 * np.sqrt(nn)
        for k, L in looks.items():
            rej[k] += z[:, L - 1].any(axis=1).sum()
        logM = nn * math.log(2) + gammaln(S + 1) + gammaln(nn - S + 1) - gammaln(nn + 2)
        hit = logM >= math.log(20)
        bet_rej += hit.any(axis=1).sum()
        first = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, np.inf)
        stop_times.extend(first.tolist())
    return {k: v / runs for k, v in rej.items()}, bet_rej / runs, np.array(stop_times)
z_rates, bet_rate, _ = simulate(0.5)
print("null hypothesis true (fair coin), 10000 pairs:")
for k, v in z_rates.items(): print(f"   classical test, looking {k:10s}: false alarm rate {v:.3f}")
print(f"   betting test (alpha = 0.05), looking after every pair: false alarm rate {bet_rate:.3f}")
_, power, T = simulate(0.55, runs=2000)
print(f"\ntheta = 0.55: betting test rejects within 10000 pairs with probability {power:.3f}; "
      f"stopping time median {np.median(T):.0f}, 90% quantile {np.quantile(T, 0.9):.0f}")
th = 0.55
nfix = math.ceil(((1.96 + 0.8416) * 2 * math.sqrt(th * (1 - th)) / (2 * th - 1))**2)
print("fixed sample size for 80% power of the classical test:", nfix)

With a single look at $n=10\,000$ the classical test would err $5\%$ of the time. Looking every $1000$ pairs raises the false alarm rate to about $20\%$, looking every $100$ pairs to about $38\%$, and looking after every pair to about $65\%$; with an unlimited experiment it would tend to $100\%$ (the law of the iterated logarithm). The betting test, checked after every single pair, stays below $5\%$, as Ville's inequality guarantees (in fact well below, since the bound is not attained). Its price shows under the alternative: the classical test with a fixed sample needs about $780$ pairs for $80\%$ power, whereas the betting test's median stopping time is larger, around $1000$ pairs, with a long right tail. In exchange, the experimenter may stop whenever the evidence is clear, and never needs to fix the sample size in advance.